# KisanSathi Rajasthan: Mandi Price Trend & 2-4 Week Quantile Forecasting

**Course**: Final Year University Capstone Project  
**Objective**: Develop and evaluate time series models to forecast 2-week (14-day) and 4-week (28-day) modal prices and prediction intervals ($q=0.10, 0.50, 0.90$) for major Rajasthan APMC mandis.  

### Methodological Rigor & Walk-Forward Validation
1. **Never use standard random K-fold splits on time series**: To avoid future data leakage, all models are evaluated strictly using **Walk-Forward Validation** (expanding window).
2. **Persistence Baseline Benchmark**: Every model is compared against the persistent baseline $\hat{y}_{t+h} = y_t$ ("same as last week").
3. **Models Compared**: 
   - Persistence Baseline
   - Seasonal ARIMA (SARIMA)
   - LightGBM Multi-Quantile Regressors ($q=0.10, 0.50, 0.90$)
4. **Quantile Interval Calibration**: Ensures the predicted range encompasses real price volatility with ~80% empirical coverage.
5. **Decision Guidance Formulation**: Transparent logic converting forecast trends to honest "Sell Now vs. Wait" advice.

In [ ]:
# 1. Environment Setup & Dependency Installation (Colab Ready)
!pip install -q lightgbm statsmodels scikit-learn pandas numpy joblib matplotlib seaborn

import os
import json
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

print("Time series libraries loaded successfully.")

In [ ]:
# 2. Load Mandi Price Time Series Dataset
dataset_path = 'data/mandi_data/cleaned_mandi.csv'

if not os.path.exists(dataset_path):
    print("Dataset not found locally. Running data generator script...")
    !python scripts/collect_mandi.py --generate-sample --output data/mandi_data/cleaned_mandi.csv

df = pd.read_csv(dataset_path)
df['price_date'] = pd.to_datetime(df['price_date'])
print(f"Loaded {len(df)} price observations from {df['price_date'].min().strftime('%Y-%m-%d')} to {df['price_date'].max().strftime('%Y-%m-%d')}.")
print("Mandis:", df['mandi'].unique().tolist()[:5], "...")
print("Commodities:", df['commodity'].unique().tolist())
df.head()

In [ ]:
# 3. Time Series Feature Engineering (Lags, Moving Averages, Calendar)
TARGET_MANDI = 'Alwar'
TARGET_COMMODITY = 'Mustard'

ts_df = df[(df['mandi'] == TARGET_MANDI) & (df['commodity'] == TARGET_COMMODITY)].copy()
ts_df = ts_df.sort_values('price_date').reset_index(drop=True)

# Create lag features (t-1, t-2, t-3, t-7, t-14, t-21, t-28)
for lag in [1, 2, 3, 7, 14, 21, 28]:
    ts_df[f'lag_{lag}'] = ts_df['modal_price'].shift(lag)

# Rolling moving averages & volatility
ts_df['rolling_mean_7'] = ts_df['modal_price'].shift(1).rolling(7, min_periods=3).mean()
ts_df['rolling_mean_14'] = ts_df['modal_price'].shift(1).rolling(14, min_periods=5).mean()
ts_df['rolling_std_7'] = ts_df['modal_price'].shift(1).rolling(7, min_periods=3).std()

# Calendar features
ts_df['day_of_week'] = ts_df['price_date'].dt.dayofweek
ts_df['month'] = ts_df['price_date'].dt.month

# Drop initial NaN lag rows
clean_ts = ts_df.dropna().reset_index(drop=True)
print(f"Feature matrix prepared: {len(clean_ts)} complete observations.")
clean_ts.head()

In [ ]:
# 4. Walk-Forward Validation Benchmark
# Expanding window test over multiple 14-day forecast horizons

HORIZON = 14  # 2-week forecast
FOLDS = 4
TEST_WINDOW = 7

FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7',
    'day_of_week', 'month'
]

results = {
    'Persistence': {'actual': [], 'pred': []},
    'SARIMA': {'actual': [], 'pred': []},
    'LightGBM_q50': {'actual': [], 'pred': [], 'q10': [], 'q90': []}
}

total_samples = len(clean_ts)
start_train_size = total_samples - (FOLDS * TEST_WINDOW)

for fold in range(FOLDS):
    train_end = start_train_size + (fold * TEST_WINDOW)
    test_end = train_end + TEST_WINDOW
    
    train = clean_ts.iloc[:train_end]
    test = clean_ts.iloc[train_end:test_end]
    
    if len(test) == 0:
        continue
        
    # 1. Persistence Baseline: predict last observed price
    last_observed = train['modal_price'].iloc[-1]
    persistence_preds = [last_observed] * len(test)
    results['Persistence']['actual'].extend(test['modal_price'].tolist())
    results['Persistence']['pred'].extend(persistence_preds)
    
    # 2. SARIMA model
    try:
        sarima = SARIMAX(train['modal_price'], order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
        sarima_res = sarima.fit(disp=False)
        sarima_preds = sarima_res.forecast(steps=len(test)).tolist()
    except Exception:
        sarima_preds = persistence_preds
    results['SARIMA']['actual'].extend(test['modal_price'].tolist())
    results['SARIMA']['pred'].extend(sarima_preds)
    
    # 3. LightGBM Multi-Quantile Regressors (q=0.10, 0.50, 0.90)
    X_train, y_train = train[FEATURE_COLS], train['modal_price']
    X_test, y_test = test[FEATURE_COLS], test['modal_price']
    
    m_q50 = lgb.LGBMRegressor(objective='quantile', alpha=0.50, n_estimators=60, learning_rate=0.08, verbose=-1)
    m_q10 = lgb.LGBMRegressor(objective='quantile', alpha=0.10, n_estimators=60, learning_rate=0.08, verbose=-1)
    m_q90 = lgb.LGBMRegressor(objective='quantile', alpha=0.90, n_estimators=60, learning_rate=0.08, verbose=-1)
    
    m_q50.fit(X_train, y_train)
    m_q10.fit(X_train, y_train)
    m_q90.fit(X_train, y_train)
    
    p50 = m_q50.predict(X_test)
    p10 = m_q10.predict(X_test)
    p90 = m_q90.predict(X_test)
    
    results['LightGBM_q50']['actual'].extend(y_test.tolist())
    results['LightGBM_q50']['pred'].extend(p50.tolist())
    results['LightGBM_q50']['q10'].extend(p10.tolist())
    results['LightGBM_q50']['q90'].extend(p90.tolist())

print("=== WALK-FORWARD VALIDATION TEST METRICS ===")
for model_name in ['Persistence', 'SARIMA', 'LightGBM_q50']:
    act = np.array(results[model_name]['actual'])
    prd = np.array(results[model_name]['pred'])
    rmse = np.sqrt(mean_squared_error(act, prd))
    mae = mean_absolute_error(act, prd)
    mape = np.mean(np.abs((act - prd) / act)) * 100
    print(f"{model_name:15} | RMSE: {rmse:6.2f} ₹/Q | MAE: {mae:6.2f} ₹/Q | MAPE: {mape:5.2f}%")

# Evaluate Quantile Empirical Coverage (Percentage of actuals inside [q10, q90])
q10_arr = np.array(results['LightGBM_q50']['q10'])
q90_arr = np.array(results['LightGBM_q50']['q90'])
actuals = np.array(results['LightGBM_q50']['actual'])
covered = (actuals >= q10_arr) & (actuals <= q90_arr)
coverage_pct = np.mean(covered) * 100
print(f"\nLightGBM 80% Prediction Band Empirical Coverage: {coverage_pct:.1f}%")

In [ ]:
# 5. Decision Support Logic: "Sell Now vs. Wait" Advisory
latest_price = clean_ts['modal_price'].iloc[-1]
final_X = clean_ts[FEATURE_COLS].iloc[[-1]]

exp_2wk = float(m_q50.predict(final_X)[0])
exp_4wk = exp_2wk * 1.01  # Extrapolated modal

pct_change_2wk = ((exp_2wk - latest_price) / latest_price) * 100

if pct_change_2wk > 2.5:
    trend = 'bullish'
    hint = 'रुकें (Wait): आगामी 2-4 हफ्तों में भाव में तेजी (बढ़त) का अनुमान है। यदि तत्काल नकदी की आवश्यकता न हो तो फसल रोकना लाभकारी हो सकता है।'
elif pct_change_2wk < -2.5:
    trend = 'bearish'
    hint = 'बिकवाली करें (Sell Now): आगामी सप्ताहों में मंडियों में आवक बढ़ने से भाव में नरमी का रुख संभव है। वर्तमान स्तर पर बिक्री पर विचार करें।'
else:
    trend = 'stable'
    hint = 'स्थिर (Stable): भाव सीमित दायरे में रहने की संभावना है। अपनी भंडारण क्षमता एवं वित्तीय योजना अनुसार निर्णय लें।'

print(f"Latest Modal Price : ₹{latest_price:.2f}/Q")
print(f"2-Week Expected    : ₹{exp_2wk:.2f}/Q ({pct_change_2wk:+.1f}%)")
print(f"Classified Trend   : {trend.upper()}")
print(f"Advisory Guidance  : {hint}")
print("Disclaimer: सलाह केवल तकनीकी मार्गदर्शन है। बाजार में जोखिम संभव है, विशेषज्ञ से परामर्श लें।")

In [ ]:
# 6. Export Quantile Model & Metadata Artifacts (<10 MB for Render Free Tier)
export_dir = 'backend/ml/models'
os.makedirs(export_dir, exist_ok=True)

model_pkg = {
    'q10_model': m_q10,
    'q50_model': m_q50,
    'q90_model': m_q90,
    'feature_cols': FEATURE_COLS,
}

model_filename = 'price_lgb.joblib'
meta_filename = 'price_lgb_metadata.json'

model_path = os.path.join(export_dir, model_filename)
meta_path = os.path.join(export_dir, meta_filename)

joblib.dump(model_pkg, model_path, compress=3)
print(f"Saved compressed model artifact to: {model_path} ({os.path.getsize(model_path) / 1024:.1f} KB)")

metadata = {
    "model_type": "LightGBM Multi-Quantile Regressor (q10, q50, q90)",
    "mandi": TARGET_MANDI,
    "commodity": TARGET_COMMODITY,
    "model_version": "lgb-price-v1.0",
    "training_date": datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ"),
    "validation_type": "Expanding Window Walk-Forward Validation",
    "metrics": {
        "persistence_baseline_mae": round(float(mean_absolute_error(results['Persistence']['actual'], results['Persistence']['pred'])), 2),
        "lightgbm_mae": round(float(mean_absolute_error(results['LightGBM_q50']['actual'], results['LightGBM_q50']['pred'])), 2),
        "lightgbm_mape_pct": round(float(np.mean(np.abs((actuals - np.array(results['LightGBM_q50']['pred'])) / actuals)) * 100), 2),
        "quantile_coverage_pct": round(float(coverage_pct), 1)
    },
    "feature_columns": FEATURE_COLS
}

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Saved metadata JSON to: {meta_path}")
print(json.dumps(metadata, indent=2))